# Chapter 6 &mdash; Karpathy's GPT as a Markov Chain, on a Jove DFA

**Concept 13 of the Chapter 6 decomposition:** *Karpathy's GPT as a Markov Chain, Learning a Jove DFA*

Andrej Karpathy's baby GPT in the raw, trained on the strings a Jove DFA accepts, and drawn as the machine it is &mdash; before and after.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter6-DFAOps/Concept-Karpathy-GPT-On-A-Jove-DFA/Concept-Karpathy-GPT-On-A-Jove-DFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


This is Andrej Karpathy's *GPT as a finite-state Markov chain*, with Jove supplying
the language.

**The transformer is in this notebook, not in a library.** About 120 lines of PyTorch,
below, for you to read and change. Nothing in it knows what an automaton is.

With `vocab_size = 2` and `context_length = 3` the model has $2^3 = 8$ possible states
&mdash; the eight three-bit windows &mdash; and Karpathy's `plot_model` draws them as a
graph with the next-bit probability written on every arrow.

The shape of the notebook:

1. read the model;
2. draw it **before** training, when every arrow is near 50%;
3. write a DFA in Jove markdown, and run together every string it accepts;
4. train, watching the loss;
5. draw it **again**, and compare;
6. sample from it, and see what kind of strings come out.

Nothing is scored. Look at the two pictures.

## 2. Definitions

### The knobs

In [ ]:
# hyperparameters for our GPT  --  CHANGE THESE
#
# vocab size is 2, so we only have two possible tokens: 0, 1
vocab_size = 2
# context length is 3, so we take 3 bits to predict the next bit probability
context_length = 3

print('state space (for this exercise) =', vocab_size ** context_length)

### The model &mdash; Karpathy's, unchanged

In [ ]:
#@title minimal GPT implementation in PyTorch  (Andrej Karpathy)
#
# Read it, change it, break it.  This is the whole model: an embedding, a
# few attention blocks, a linear head.  Nothing here knows about automata.
""" super minimal decoder-only gpt """

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k ,v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.nonlin = nn.GELU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.nonlin(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    # these are default GPT-2 hyperparameters
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    bias: bool = False

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %d" % (sum(p.nelement() for p in self.parameters()),))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, -1, :]) # note: only returning logits at the last time step (-1), output is 2D (b, vocab_size)
        return logits

### Karpathy's picture of it

In [ ]:
# --- Karpathy's picture: the model AS a finite-state machine -------------
from graphviz import Digraph

def all_possible(n, k):
    # every list of k elements, each in range(n)
    if k == 0:
        yield []
    else:
        for i in range(n):
            for c in all_possible(n, k - 1):
                yield [i] + c

def plot_model(gpt, title=''):
    dot = Digraph(comment='baby GPT', engine='circo')
    if title:
        dot.attr(label=title, labelloc='t', fontsize='12')
    for xi in all_possible(gpt.config.vocab_size, gpt.config.block_size):
        x = torch.tensor(xi, dtype=torch.long)[None, ...]
        y = nn.functional.softmax(gpt(x), dim=-1)[0].tolist()
        here = ''.join(str(d) for d in xi)
        dot.node(here)
        for t in range(gpt.config.vocab_size):
            nxt = ''.join(str(d) for d in (xi[1:] + [t]))
            dot.edge(here, nxt, label='%d(%.0f%%)' % (t, y[t] * 100),
                     color='#b3251f' if t == 0 else '#1a53c0')
    return dot

### The training data, from a DFA

In [ ]:
# --- the training data: strings the DFA accepts, run together ------------
from functools import reduce

def corpus_from(D, upto=400):
    strings = [nthnumeric(i, ['0', '1']) for i in range(upto)]
    good = [s for s in strings if accepts_dfa(D, s)]
    return good, list(map(int, reduce(lambda a, b: a + b, good)))

def make_XY(seq, context_length):
    X, Y = [], []
    for i in range(len(seq) - context_length):
        X.append(seq[i:i + context_length])
        Y.append(seq[i + context_length])
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long))

def train_gpt(gpt, X, Y, iters=200, lr=1e-3, every=20):
    optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, weight_decay=1e-1)
    losses = []
    for i in range(iters):
        logits = gpt(X)
        loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
        if i % every == 0 or i == iters - 1:
            print(i, loss.item())
    return losses

### The language

In [ ]:
# --- the language, as a Jove DFA  --  CHANGE THIS ------------------------
# No two 1s in a row.  A good first language because the constraint is
# LOCAL: after a 1, a 1 is forbidden, and three bits of context is plenty
# to see that.  Try ENDS01 below once you have seen this one work.
NO11 = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> F1
F1 : 0 -> IF
F1 : 1 -> D
D  : 0|1 -> D
''')

ENDS01 = md2mc('''DFA
I  : 0 -> S0
I  : 1 -> I
S0 : 0 -> S0
S0 : 1 -> F
F  : 0 -> S0
F  : 1 -> I
''')

LANG = NO11                      # <-- change me
dotObj_dfa(min_dfa(LANG), FuseEdges=True)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;12.&nbsp;DeMorgan's Law for DFA, Verified by Isomorphism](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter6-DFAOps/Concept-DeMorgan-For-DFA/Concept-DeMorgan-For-DFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;14.&nbsp;Changing the Language: Which Ones Does It Pick Up?](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter6-DFAOps/Concept-Changing-The-Language/Concept-Changing-The-Language.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Before training.** The weights are random, so every arrow is near a half. Eight states already; training will move the arrows, not make more states.

In [ ]:
config = GPTConfig(block_size=context_length, vocab_size=vocab_size,
                   n_layer=4, n_head=4, n_embd=16, bias=False)
torch.manual_seed(1337)
gpt = GPT(config)
plot_model(gpt, 'before training')

**The corpus**: every string the DFA accepts, up to a length, run together.

In [ ]:
good, seq = corpus_from(LANG)
print('accepted strings :', len(good))
print('first few        :', [s or 'eps' for s in good[:8]])
print('training sequence:', ''.join(map(str, seq))[:60], '...')
print('length           :', len(seq))

Cut it into examples: three bits in, the next bit out.

In [ ]:
X, Y = make_XY(seq, context_length)
for i in range(6):
    print('example %2d: %s --> %s' % (i + 1, X[i].tolist(), Y[i].item()))
print('...')
print(X.shape, Y.shape)

**Train**, and watch the loss fall.

In [ ]:
losses = train_gpt(gpt, X, Y, iters=200, every=20)

**After training.** Same eight states. Compare the arrows with the picture above, and with the DFA.

In [ ]:
plot_model(gpt, 'after training')

What changed. Every window ending in 1 now sends almost nothing to 1 &mdash; the model has picked up the rule. The few per cent that remain are not noise; see exercise 4.

In [ ]:
import itertools
print('window   P(1) after training')
for w in itertools.product([0, 1], repeat=context_length):
    x = torch.tensor(list(w), dtype=torch.long)[None, ...]
    p = nn.functional.softmax(gpt(x), dim=-1)[0].tolist()
    print('  %s       %.2f' % (''.join(map(str, w)), p[1]))

**Sample from it.** Give it three bits to start and let it continue.

In [ ]:
# --- sample from the model, exactly as Karpathy does --------------------
def sample(gpt, start, steps=24):
    xi = list(start)
    full = xi.copy()
    for _ in range(steps):
        x = torch.tensor(xi, dtype=torch.long)[None, ...]
        probs = nn.functional.softmax(gpt(x), dim=-1)
        t = torch.multinomial(probs[0], num_samples=1).item()
        xi = xi[1:] + [t]
        full.append(t)
    return ''.join(map(str, full))


for start in ([0, 0, 1], [1, 1, 1]):
    print(start, '->', sample(gpt, start))

## 4. Exercises


1. Set `LANG = ENDS01` and re-run. The arrows barely move. Why is that language
   harder for this model than "no two 1s in a row"?
2. Change `context_length` to 2 and re-run. How many states now, and does the picture
   still look like the DFA?
3. Train for 20 iterations instead of 200. Which arrows settle first?
4. In `corpus_from`, lower `upto` from 400 to 60. What happens to the picture, and why?
4. After a window ending in 1 the model still gives 1 about 7% of the probability,
   even though the language forbids it outright. Where does that 7% come from? (The
   accepted strings were run together end to end.)
5. Read the `CausalSelfAttention` class. Which line makes it *causal*, and what would
   break if you deleted it?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-Karpathy-GPT-On-A-Jove-DFA')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')